<a href="https://colab.research.google.com/github/nidhinvijay/AI_Interview_Bot/blob/main/Audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!pip install -U openai-whisper
!pip install transformers torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 15.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=36de9733ba3311638a2eef5f2dbfe9b623bb2eedf05986207c76f5006560e6a2
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [2]:

import whisper
import json
from transformers import pipeline
import torch
import time
from google.colab import files
import os

def format_timestamp(seconds):
    """Converts seconds into HH:MM:SS or MM:SS format."""
    t = time.gmtime(seconds)
    if seconds >= 3600:
        return time.strftime('%H:%M:%S', t)
    else:
        return time.strftime('%M:%S', t)

In [3]:

def analyze_audio(audio_file_path):
    # Check if a GPU is available
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"✅ System check: Using device '{DEVICE}'")

    # 1. Transcription
    print("\nLoading Whisper model...")
    model = whisper.load_model("small", device=DEVICE)
    print(f"🎤 Starting transcription for '{audio_file_path}'...")
    transcription_result = model.transcribe(audio_file_path, word_timestamps=True)
    print("Transcription finished.")

    # 2. Toxicity Analysis
    print("\nLoading toxicity analysis model...")
    toxicity_classifier = pipeline(
        "text-classification",
        model="unitary/toxic-bert",
        tokenizer="bert-base-uncased",
        device=0 if DEVICE == "cuda" else -1
    )
    final_analysis = []
    print("🔍 Analyzing transcribed segments for toxicity...")
    for segment in transcription_result["segments"]:
        text = segment["text"].strip()
        if not text:
            continue
        scores = {r['label']: r['score'] for r in toxicity_classifier(text)}
        final_analysis.append({
            "sentence": text,
            "start_time": round(segment["start"], 2),
            "end_time": round(segment["end"], 2),
            "toxicity_scores": scores
        })
    print("Toxicity analysis finished.")
    return final_analysis

In [4]:

# 1. Upload the file
print("Please upload your audio file...")
uploaded = files.upload()

# Check if a file was uploaded
if not uploaded:
    print("\nNo file uploaded. Please run the cell again.")
else:
    # Get the name of the uploaded file
    audio_file_path = list(uploaded.keys())[0]
    print(f"\nFile '{audio_file_path}' uploaded successfully.")

    # 2. Run the full analysis
    full_analysis = analyze_audio(audio_file_path)

    # 3. Save the full JSON output
    json_filename = "audio_analysis_output.json"
    with open(json_filename, "w", encoding="utf-8") as f:
        json.dump(full_analysis, f, indent=4)
    print(f"\n✅ Full analysis saved to '{json_filename}'")

    # 4. Create and save the disrespectful summary
    summary_filename = "disrespectful_summary.txt"
    flagged_sentences = []
    for item in full_analysis:
        scores = item['toxicity_scores']
        if (scores.get('toxic', 0) > 0.7 or
            scores.get('insult', 0) > 0.7 or
            scores.get('threat', 0) > 0.7):
            flagged_sentences.append(item)

    # Write the summary and print to console
    print(f"\n--- Analysis Summary ---")
    if not flagged_sentences:
        message = "👍 No highly disrespectful sentences were detected."
        print(message)
        with open(summary_filename, "w", encoding="utf-8") as f:
            f.write(message)
    else:
        message = f"🚨 Found {len(flagged_sentences)} potentially disrespectful sentences. Summary saved to '{summary_filename}'"
        print(message)
        with open(summary_filename, "w", encoding="utf-8") as f:
            f.write("Potentially Disrespectful Sentences Detected\n" + "="*40 + "\n\n")
            for item in flagged_sentences:
                start_formatted = format_timestamp(item['start_time'])
                print(f"🚨 [{start_formatted}]: \"{item['sentence']}\"")
                f.write(f"Timestamp: {start_formatted}\nSentence: \"{item['sentence']}\"\n\n")

    # 5. Provide download links
    print("\n📄 You can download the result files from the file browser on the left.")

Please upload your audio file...


Saving my_audio.mp3.wav to my_audio.mp3.wav

File 'my_audio.mp3.wav' uploaded successfully.
✅ System check: Using device 'cuda'

Loading Whisper model...


100%|███████████████████████████████████████| 461M/461M [00:09<00:00, 49.8MiB/s]


🎤 Starting transcription for 'my_audio.mp3.wav'...
Transcription finished.

Loading toxicity analysis model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


🔍 Analyzing transcribed segments for toxicity...
Toxicity analysis finished.

✅ Full analysis saved to 'audio_analysis_output.json'

--- Analysis Summary ---
🚨 Found 7 potentially disrespectful sentences. Summary saved to 'disrespectful_summary.txt'
🚨 [00:34]: "You think Garga gives a shit about you?"
🚨 [00:37]: "He gives a shit about his money."
🚨 [01:30]: "You fucking Judas!"
🚨 [01:34]: "Okay, I'm a fucking Judas."
🚨 [12:40]: "You ancient fuck."
🚨 [12:46]: "I'm going to eat your brain with a spoon."
🚨 [12:49]: "I'm going to eat your fucking brain with a spoon."

📄 You can download the result files from the file browser on the left.
